# NB05 — ConvNeXt-Tiny Feature Extraction

Extracts 768-d patch features at both scales (0.5 and 2.0 μm/pixel) using ImageNet-pretrained ConvNeXt-Tiny. Features are saved per-slide as `{slide_id}.npy` plus `{slide_id}_meta.parquet` under `features/scale0p5/` and `features/scale2p0/`.

Includes a 60-second self-test that gates the full run; below the throughput target (50 tiles/s) the cell aborts. The model class is parameterized by `trainable` so NB06 can reuse the same definition with `trainable=True` for backbone fine-tuning.

In [ ]:
import os, sys, json, time, math, random, subprocess, platform, gc
from pathlib import Path
from datetime import datetime
from time import perf_counter

WORKSPACE = Path(os.environ.get('WORKSPACE', './workspace'))
WSI_ROOT  = Path(os.environ.get('WSI_ROOT',  './data/wsi'))
SUBDIRS = {
    'features':  WORKSPACE / 'features',
    'tiles':     WORKSPACE / 'tiles',
    'logs':      WORKSPACE / 'logs',
    'figures':   WORKSPACE / 'figures',
    'manifests': WORKSPACE / 'manifests',
}
for p in SUBDIRS.values():
    p.mkdir(parents=True, exist_ok=True)

TSUM = SUBDIRS['tiles'] / 'tiling_summary_tcga.parquet'
assert TSUM.exists(), f'missing tiling summary: {TSUM} (run NB04 first)'

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as tvm
import torchvision.transforms as T
from PIL import Image
import openslide

DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
AMP_DTYPE = torch.float16 if DEVICE == 'cuda' else torch.bfloat16
TILE_SIZE = 256; MODEL_IN = 224; SELFTEST_SECONDS = 60
TARGET_TILES_PER_SEC = 50.0; RANDOM_SEED = 13; SAVE_DTYPE = np.float16

random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)
if hasattr(torch.backends, 'cudnn'):
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.allow_tf32 = True
if hasattr(torch, 'set_float32_matmul_precision'):
    torch.set_float32_matmul_precision('high')

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
_to_tensor = T.ToTensor()
_resize    = T.Resize((MODEL_IN, MODEL_IN), interpolation=T.InterpolationMode.BILINEAR)
_normalize = T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)

def to_model_tensor(img: Image.Image) -> torch.Tensor:
    if img.size != (MODEL_IN, MODEL_IN):
        img = _resize(img)
    t = _to_tensor(img)
    t = _normalize(t)
    return t

class ConvNeXtTinyFeats(nn.Module):
    def __init__(self, trainable: bool = False):
        super().__init__()
        w = tvm.ConvNeXt_Tiny_Weights.DEFAULT
        m = tvm.convnext_tiny(weights=w)
        self.features = m.features
        self.gap = nn.AdaptiveAvgPool2d(1)
        for p in self.parameters():
            p.requires_grad = trainable
        if not trainable:
            self.eval()
    def forward(self, x):
        x = self.features(x)
        x = self.gap(x).flatten(1)
        return x

def build_model():
    m = ConvNeXtTinyFeats(trainable=False).to(DEVICE)
    if DEVICE == 'cuda':
        m = m.to(memory_format=torch.channels_last)
        d = torch.randn(256, 3, MODEL_IN, MODEL_IN, device=DEVICE).to(memory_format=torch.channels_last)
        with torch.no_grad(), torch.amp.autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=True):
            _ = m(d)
        torch.cuda.synchronize()
    return m

MODEL = build_model()

df_sum = pd.read_parquet(TSUM)
assert 'manifest' in df_sum.columns and 'slide_id' in df_sum.columns

SLIDE_INDEX_PATH = SUBDIRS['logs'] / 'slide_path_index.json'
def index_slide_paths(root: Path) -> dict:
    print('[INDEX] building slide path map (one-time)')
    mp = {}
    for ext in ('*.svs', '*.ndpi', '*.tif', '*.mrxs', '*.scn'):
        for p in root.rglob(ext):
            mp[p.stem] = str(p)
    return mp

if SLIDE_INDEX_PATH.exists():
    slide_map = json.loads(SLIDE_INDEX_PATH.read_text(encoding='utf-8'))
else:
    slide_map = index_slide_paths(WSI_ROOT)
    SLIDE_INDEX_PATH.write_text(json.dumps(slide_map, indent=2), encoding='utf-8')

def slide_path_from_id(slide_id, manifest_df=None):
    if manifest_df is not None:
        for cand in ('path', 'source_path', 'slide_path', 'wsi_path'):
            if cand in manifest_df.columns:
                p = manifest_df[cand].iloc[0]
                if isinstance(p, str) and Path(p).exists():
                    return p
    if slide_id in slide_map:
        return slide_map[slide_id]
    base = slide_id.split('.')[0]
    return slide_map.get(base, None)

def load_manifest(man_path):
    m = pd.read_parquet(man_path)
    lower = {c.lower(): c for c in m.columns}
    def pick(*names):
        for n in names:
            if n in m.columns: return n
            if n.lower() in lower: return lower[n.lower()]
        raise KeyError(f'missing columns {names} in {man_path.name}')
    xcol = pick('x', 'px_x', 'x_level')
    ycol = pick('y', 'px_y', 'y_level')
    lvlcol = pick('level', 'lvl')
    tsize = TILE_SIZE
    for n in ('tile_size', 'tile_px', 'size'):
        if n in m.columns:
            try: tsize = int(m[n].iloc[0])
            except Exception: pass
            break
    return m, xcol, ycol, lvlcol, tsize

class SlideReader:
    def __init__(self, path):
        self.path = path
        self.osr = openslide.OpenSlide(path)
        self.down = list(self.osr.level_downsamples)
    def read_tile(self, level, x_level, y_level, size):
        ds = self.down[level]
        bx = int(round(x_level * ds))
        by = int(round(y_level * ds))
        return self.osr.read_region((bx, by), level, (size, size)).convert('RGB')
    def close(self):
        try: self.osr.close()
        except Exception: pass

def iter_batches_from_manifest(reader, man_df, xcol, ycol, lvlcol, tile_px, max_batch=4096):
    buf = []
    for r in man_df[[xcol, ycol, lvlcol]].itertuples(index=False, name=None):
        x, y, lvl = map(int, r)
        img = reader.read_tile(lvl, x, y, tile_px)
        buf.append(to_model_tensor(img))
        if len(buf) >= max_batch:
            yield torch.stack(buf, 0).to(memory_format=torch.channels_last)
            buf.clear()
    if buf:
        yield torch.stack(buf, 0).to(memory_format=torch.channels_last)

def forward_batches(model, batches_iter):
    outs = []
    for cpu_batch in batches_iter:
        with torch.no_grad():
            chunk = cpu_batch.to(DEVICE, non_blocking=True)
            with torch.amp.autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=(DEVICE == 'cuda')):
                out = model(chunk)
            outs.append(out.detach().cpu())
        del cpu_batch
    return torch.cat(outs, 0).contiguous().numpy()

OUT05 = SUBDIRS['features'] / 'scale0p5'
OUT20 = SUBDIRS['features'] / 'scale2p0'
OUT05.mkdir(parents=True, exist_ok=True)
OUT20.mkdir(parents=True, exist_ok=True)

def out_paths(slide_id, scale):
    d = OUT05 if math.isclose(scale, 0.5, abs_tol=1e-6) else OUT20
    return d / f'{slide_id}.npy', d / f'{slide_id}_meta.parquet'

env_s5 = {
    'time': datetime.now().isoformat(timespec='seconds'),
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'device': DEVICE, 'torch': torch.__version__, 'amp_dtype': str(AMP_DTYPE),
}
(SUBDIRS['logs'] / 'nb05_env.json').write_text(json.dumps(env_s5, indent=2), encoding='utf-8')
print(json.dumps(env_s5, indent=2))

done_map = {}
for sc in (0.5, 2.0):
    sub = df_sum[np.isclose(df_sum['scale_um_per_px'], sc)]
    for sid in sub['slide_id'].unique():
        npy, meta = out_paths(sid, sc)
        done_map[(sid, sc)] = npy.exists() and meta.exists()

groups = []
for sid, g in df_sum.groupby('slide_id', sort=False):
    entries = []
    for _, row in g.sort_values('n_tiles', ascending=False).iterrows():
        sc = float(row['scale_um_per_px'])
        if not done_map.get((sid, sc), False):
            entries.append({'scale': sc, 'manifest': Path(row['manifest'])})
    if entries:
        groups.append({'slide_id': sid, 'entries': entries})
print(f'[INFO] slides pending (≥1 scale): {len(groups)}')

def selftest(seconds=SELFTEST_SECONDS, target=TARGET_TILES_PER_SEC):
    cand = []
    for sid, g in df_sum.groupby('slide_id'):
        n = int(g['n_tiles'].sum())
        manp = Path(g.sort_values('n_tiles').iloc[-1]['manifest'])
        cand.append((n, sid, manp))
    cand.sort(key=lambda x: x[0])
    pick = cand[:min(12, len(cand))]
    readers = {}
    for _, sid, manp in pick:
        m, xcol, ycol, lvlcol, tpx = load_manifest(manp)
        fn = slide_path_from_id(sid, m)
        if not fn or not Path(fn).exists():
            continue
        readers[sid] = (SlideReader(fn), m[[xcol, ycol, lvlcol]].copy(), xcol, ycol, lvlcol, tpx)
    tiles_done = 0; t0 = perf_counter(); stop = t0 + seconds
    while perf_counter() < stop and readers:
        for sid, (sr, m, xcol, ycol, lvlcol, tpx) in list(readers.items()):
            take = m.iloc[:512]
            if take.empty:
                del readers[sid]; sr.close(); continue
            batches = iter_batches_from_manifest(sr, take, xcol, ycol, lvlcol, tpx, max_batch=2048)
            with torch.no_grad():
                for cpu_batch in batches:
                    chunk = cpu_batch.to(DEVICE, non_blocking=True)
                    with torch.amp.autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=(DEVICE == 'cuda')):
                        _ = MODEL(chunk)
                    tiles_done += chunk.size(0)
                    del cpu_batch, chunk
                    if perf_counter() >= stop: break
            m = m.iloc[len(take):]
            readers[sid] = (sr, m, xcol, ycol, lvlcol, tpx)
            if perf_counter() >= stop: break
    dt = perf_counter() - t0
    rate = tiles_done / max(dt, 1e-6)
    print(f'[SELFTEST] tiles={tiles_done} time={dt:.1f}s rate={rate:.1f} tiles/s')
    print(('[PASS] ' if rate >= target else '[FAIL] ') + f'{rate:.1f} tiles/s (target ≥ {target:.0f})')
    (SUBDIRS['logs'] / 'nb05_selftest.json').write_text(json.dumps({
        'tiles': tiles_done, 'seconds': round(dt, 2),
        'tiles_per_s': round(rate, 2), 'target': target, 'pass': rate >= target,
    }, indent=2), encoding='utf-8')
    for sid, (sr, *_rest) in readers.items():
        sr.close()
    return rate

rate = selftest()
if rate < TARGET_TILES_PER_SEC:
    print('[ABORT] below target; not running full extraction')
    raise SystemExit(0)

PROG = SUBDIRS['logs'] / 'nb05_progress.jsonl'
def log_progress(**kw):
    kw['ts'] = datetime.now().isoformat(timespec='seconds')
    with open(PROG, 'a', encoding='utf-8') as f:
        f.write(json.dumps(kw, ensure_ascii=False) + '\n')

for i, grp in enumerate(groups, 1):
    sid = grp['slide_id']
    man_pref = min(grp['entries'], key=lambda e: abs(e['scale'] - 0.5))
    m_probe, xcol, ycol, lvlcol, tpx = load_manifest(man_pref['manifest'])
    fn = slide_path_from_id(sid, m_probe)
    if not fn or not Path(fn).exists():
        print(f'[WARN] slide path not found: {sid} — skipped')
        continue
    reader = SlideReader(fn)

    for e in grp['entries']:
        sc = float(e['scale'])
        npy_path, meta_path = out_paths(sid, sc)
        if npy_path.exists() and meta_path.exists():
            continue
        man_df, xcol, ycol, lvlcol, tpx = load_manifest(e['manifest'])
        if man_df.empty:
            print(f"[WARN] empty manifest: {e['manifest']}, skip")
            continue
        t0 = perf_counter()
        batches = iter_batches_from_manifest(reader, man_df, xcol, ycol, lvlcol, tpx, max_batch=4096)
        feats = forward_batches(MODEL, batches)
        if DEVICE == 'cuda': torch.cuda.synchronize()
        dt = perf_counter() - t0
        np.save(npy_path, feats.astype(SAVE_DTYPE))
        md = man_df.copy()
        md['slide_id'] = sid; md['scale_um_per_px'] = sc
        md.to_parquet(meta_path, index=False)
        N = int(feats.shape[0])
        tiles_per_s = N / max(dt, 1e-6)
        vram = (torch.cuda.max_memory_allocated()/(1024**3)) if DEVICE == 'cuda' else 0.0
        print(f'[OK] {i}/{len(groups)} | {sid} @{sc:.1f} μm/px → ({N},768) | {tiles_per_s:.1f} tiles/s | VRAM~{vram:.2f} GB')
        log_progress(slide_id=sid, scale=sc, tiles=N, seconds=round(dt, 2),
                     tps=round(tiles_per_s, 2), vram_gb=round(vram, 2))
        del feats; gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()
    reader.close()

print('[DONE] NB05 complete. Next: NB06 (BYOL+MFR pretraining with backbone fine-tuning).')